#Bronze to Silver
## CDM → Unity Catalog: Incremental Ingestion via Auto Loader

**Migration summary (legacy → modern)**

| Legacy | Modern |
|---|---|
| Service principal credentials in notebook | UC External Location + Storage Credential (no secrets in code) |
| `cdm_to_delta` blob-copy library | Native Auto Loader (`cloudFiles`) reads directly from UC Volume |
| Manual log table to track processed blobs | Auto Loader checkpoint files (exactly-once guarantee) |
| Two-step: copy blobs → convert to Parquet | Single step: CSV Volume → Delta table |

**Pre-requisite:** An External Location must be registered in Unity Catalog pointing to the ADLS `dataflow-cdm` container, with an attached Storage Credential. The Volume at `cdm_volume_path` below should resolve to that location.

### Step 1 — Configuration
Reads runtime values from notebook parameters (widgets) so the same notebook can be reused across environments or scheduled with different inputs via a Lakeflow Job task. Parameters:

| Parameter | Purpose |
|---|---|
| `target_catalog` / `target_schema` | Target Unity Catalog location (`catalog.schema`) for ingested Delta tables |
| `entities` | Comma-separated list of CDM entity names to process (must match `model.json`) |
| `cdm_volume_path` | UC Volume path for the source CDM data (backed by an External Location — no credentials in code) |
| `checkpoint_volume_path` | UC Volume path where Auto Loader stores per-entity checkpoint state |

In [0]:
# ── Target Unity Catalog location ───────────────────────────────────────────
target_catalog = dbutils.widgets.get("target_catalog")
target_schema  = dbutils.widgets.get("target_schema")

# ── Entities to ingest (must match names in model.json) ──────────────────────
entities = [e.strip() for e in dbutils.widgets.get("entities").split(",")]

# ── UC Volume paths ───────────────────────────────────────────────────────────
# Source: External Volume backed by the ADLS 'dataflow-cdm' container.
# No credentials needed here — the UC External Location handles ADLS auth.
cdm_volume_path        = dbutils.widgets.get("cdm_volume_path")
checkpoint_volume_path = dbutils.widgets.get("checkpoint_volume_path")

print(f"Target : {target_catalog}.{target_schema}")
print(f"Source : {cdm_volume_path}")
print(f"Entities: {entities}")

Target : silver.dataverse
Source : /Volumes/bronze/default/dataverse
Entities: ['conversationtranscript', 'systemuser']


### Step 2 — Parse the CDM Manifest
Reads `model.json` from the source Volume to discover each entity’s attribute names and data types. A `CDM_TYPE_MAP` dictionary converts Microsoft CDM types (`guid`, `datetime`, `datetimeoffset`, `decimal`, `int64`, `boolean`, etc.) into their Spark equivalents, producing a typed `StructType` for each entity listed in the `entities` parameter. Supplying an explicit schema avoids Auto Loader schema inference at read time and ensures type fidelity with the upstream Dataverse export.

Raises `ValueError` if any requested entity is not found in `model.json`, printing the available entity names for troubleshooting.

In [0]:
import json
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, ShortType,
    DoubleType, FloatType, DecimalType,
    BooleanType, TimestampType, DateType, BinaryType,
)

# CDM dataType → Spark DataType
CDM_TYPE_MAP: dict = {
    "guid":           StringType(),
    "string":         StringType(),
    "int64":          LongType(),
    "int32":          IntegerType(),
    "smallinteger":   ShortType(),
    "decimal":        DecimalType(18, 6),
    "double":         DoubleType(),
    "float":          FloatType(),
    "boolean":        BooleanType(),
    "datetime":       TimestampType(),
    "datetimeoffset": TimestampType(),
    "date":           DateType(),
    "time":           StringType(),
    "binary":         BinaryType(),
}

def cdm_type_to_spark(cdm_type: str):
    return CDM_TYPE_MAP.get((cdm_type or "string").lower(), StringType())

# Read model.json from the UC Volume (no ADLS SDK needed)
model_json_path = f"{cdm_volume_path}/model.json"
with open(model_json_path) as f:
    model = json.load(f)

entity_schemas: dict[str, StructType] = {}
for entity_def in model.get("entities", []):
    name = entity_def.get("name", "")
    if name in entities:
        fields = [
            StructField(attr["name"], cdm_type_to_spark(attr.get("dataType")), nullable=True)
            for attr in entity_def.get("attributes", [])
        ]
        entity_schemas[name] = StructType(fields)
        print(f"  '{name}': {len(fields)} columns parsed from model.json")

missing = set(entities) - set(entity_schemas)
if missing:
    available = [e["name"] for e in model.get("entities", [])]
    raise ValueError(f"Entities not found in model.json: {missing}. Available: {available}")

print(f"\nSchemas ready for: {list(entity_schemas)}")

  'conversationtranscript': 50 columns parsed from model.json
  'systemuser': 169 columns parsed from model.json

Schemas ready for: ['conversationtranscript', 'systemuser']


### Step 3 — Ensure Target Catalog & Schema Exist
Idempotently creates the target catalog and schema in Unity Catalog using `IF NOT EXISTS` guards, so repeated runs are safe. Individual Delta tables are created automatically by `toTable()` in the next step on the first run — this cell only ensures their parent namespace is in place.

In [0]:
# Create the target catalog and schema if they do not already exist.
# toTable() below will create individual Delta tables on first run.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

print(f"Target schema ready: {target_catalog}.{target_schema}")

Target schema ready: silver.dataverse


### Step 4 — Schema Verification, Optional Reset, and Incremental Ingestion

**Step 4a — Schema Verification** (`conversationtranscript`): Before running the stream, a batch read with the same CSV parser options is run against the `conversationtranscript` source folder. It prints the full CDM schema (column names and Spark types), reads a 5-row sample, and displays `Id`, `content`, `conversationstarttime`, and `conversationtranscriptid` to confirm the JSON `content` column is parsed as a single field and does not spill into adjacent columns. Re-run this cell whenever source data or CSV format changes.

**Step 4b — Reset** _(run only when re-ingesting from scratch)_: Deletes each entity's Auto Loader checkpoint directory and truncates the corresponding target Delta table. Use this after changing CSV parser settings, or when upstream data must be fully re-processed. Safely skips entities whose checkpoint or table does not yet exist.

**Step 4c — Incremental Ingestion**: For each entity, Auto Loader (`cloudFiles`) scans the source Volume for new CSV partition files and processes only files not seen in prior runs, tracked via a per-entity checkpoint. Key design choices:

* **`trigger(availableNow=True)`** — processes all pending files then stops, giving batch-job semantics safe to call from a scheduled Lakeflow job.
* **`header=false` + explicit schema** — CDM partition CSVs have no column headers; the schema from `model.json` is applied positionally, avoiding inference overhead.
* **`pathGlobFilter="*.csv"`** — restricts file discovery to CSV partition files only, excluding CDM manifest JSON files (`model.json`, `*.cdm.json`) that share the same directory.
* **`multiLine=true`** — handles CSV records whose `content` JSON payload spans multiple lines.
* **`quote='"'` / `escape='"'`** — Dataverse CSV encoding: fields containing commas or quotes are outer-quoted with `"`, and inner double-quotes are escaped by doubling (`""`) rather than backslash-escaping.
* **`cloudFiles.schemaLocation`** — persists the provided schema to the checkpoint directory, allowing Auto Loader to track schema evolution across runs.
* **`_ingested_at` / `_source_file` columns** — add load timestamp and source partition path (via `_metadata.file_path`) for downstream lineage and debugging.
* **`mergeSchema=true`** — allows the target Delta table to accommodate new columns if the source schema evolves.
* **`toTable(target_table)`** — writes directly to a Unity Catalog Delta table; creates it on the first run.

In [0]:
# ── Schema verification for conversationtranscript before Auto Loader ────────
# Validates that the CSV parser keeps the JSON `content` column intact instead
# of spilling it into later columns.
ENTITY_TO_VERIFY = "conversationtranscript"

schema   = entity_schemas[ENTITY_TO_VERIFY]
src_path = f"{cdm_volume_path}/{ENTITY_TO_VERIFY}"

# 1 — Print CDM schema
print(f"CDM schema for '{ENTITY_TO_VERIFY}' ({len(schema.fields)} columns):")
for fld in schema.fields:
    print(f"  {fld.name:<55} {fld.dataType.simpleString()}")

# 2 — Sample read with the CSV quote/escape rules that Dataverse exports use.
# CDM partition CSVs have no column headers, and the JSON content field is a
# quoted CSV field with doubled double-quotes inside it.
print("\nSample (5 rows) with Dataverse CSV parsing rules applied:")
sample_df = (
    spark.read
        .format("csv")
        .option("header", "false")
        .option("pathGlobFilter", "*.csv")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .schema(schema)
        .load(src_path)
        .limit(5)
)

# 3 — Focus the verification on the columns that were previously misaligned.
verification_df = sample_df.select(
    "Id",
    "content",
    "conversationstarttime",
    "conversationtranscriptid"
)

display(verification_df)

CDM schema for 'conversationtranscript' (50 columns):
  Id                                                      string
  SinkCreatedOn                                           timestamp
  SinkModifiedOn                                          timestamp
  statecode                                               bigint
  statuscode                                              bigint
  bot_conversationtranscriptid                            string
  bot_conversationtranscriptid_entitytype                 string
  createdby                                               string
  createdby_entitytype                                    string
  createdonbehalfby                                       string
  createdonbehalfby_entitytype                            string
  modifiedby                                              string
  modifiedby_entitytype                                   string
  modifiedonbehalfby                                      string
  modifiedonbehalfby_entitytyp

Id content conversationstarttime conversationtranscriptid 9fb82d7c-469f-4a3b-ae3a-a20c97cfb425 {"activities":[{"valueType":"ConversationInfo","type":"trace","timestamp":1781276744,"timestampMs":1781276744301,"from":{"id":"","role":0},"value":{"lastSessionOutcome":"None","lastSessionOutcomeReason":"NoError","isDesignMode":true,"locale":"en-US"}},{"id":"bdf8f47f-12a1-4c89-9dda-8ea655d553e9","type":"event","timestamp":1781276744,"timestampMs":1781276744301,"from":{"id":"d77e50ca-e5f8-b42f-7d80-881be0485da1","aadObjectId":"e9c36c90-108f-4285-aa64-42d5f1b80a8e","role":1},"name":"startConversation","channelId":"pva-studio","attachments":[]},{"id":"f62dd9dd-3b8f-490c-80ec-49e0f2c2fb65","type":"message","timestamp":1781276744,"timestampMs":1781276744767,"from":{"id":"510b75bb-1d9d-3e98-1174-0e7da82412e8","role":0},"channelId":"pva-studio","textFormat":"markdown","text":"Hello, I\u0027m F1 Databricks Agent, a virtual assistant. Just so you are aware, I sometimes use AI to answer your questions. If you provided a website during creation, try asking me about it! Next try giving me some more knowledge by setting up generative AI.","attachments":[],"replyToId":"bdf8f47f-12a1-4c89-9dda-8ea655d553e9","speak":"Hello and thank you for calling F1 Databricks Agent. Please note that some responses are generated by AI and may require verification for accuracy. How may I help you today?","channelData":{"feedbackLoop":{"type":"default"}}},{"id":"39634b45-9f93-4483-b9f1-5657706dd859","type":"event","timestamp":1781276744,"timestampMs":1781276744767,"from":{"id":"510b75bb-1d9d-3e98-1174-0e7da82412e8","role":0},"name":"DialogTracing","channelId":"pva-studio","attachments":[],"replyToId":"bdf8f47f-12a1-4c89-9dda-8ea655d553e9","value":{"actions":[{"actionId":"sendMessage_M0LuhV","topicId":"auto_agent_kQU7a.topic.ConversationStart","triggerId":"main","dialogComponentId":"fc6f0497-23fc-46e9-a52d-6705e88d0c6d","actionType":"SendActivity","conditionItemExit":[],"variableState":{"dialogState":{},"globalState":{}},"exception":"","resultTrace":{}}]}},{"id":"0711aa74-3baf-4c4c-b93b-31616947f820","type":"message","timestamp":1781276856,"timestampMs":1781276856765,"from":{"id":"d77e50ca-e5f8-b42f-7d80-881be0485da1","aadObjectId":"e9c36c90-108f-4285-aa64-42d5f1b80a8e","role":1},"channelId":"pva-studio","textFormat":"plain","text":"Who won the Monte Carlo Grand Prix this year?","attachments":[],"channelData":{"attachmentSizes":[],"enableDiagnostics":true,"testMode":"Text","clientActivityID":"cz8cznnb8sc"}},{"id":"14fa343b-f68f-4281-8f42-ef78d58bab3d","type":"message","timestamp":1781276856,"timestampMs":1781276856862,"from":{"id":"510b75bb-1d9d-3e98-1174-0e7da82412e8","role":0},"name":"connectors/consentCard","channelId":"pva-studio","attachments":[{"contentType":"application/vnd.microsoft.card.adaptive","content":{"type":"AdaptiveCard","version":"1.3","body":[{"type":"TextBlock","size":"medium","weight":"bolder","text":"Connect to continue","wrap":true},{"type":"TextBlock","text":"I\u0027ll use your credentials to connect and to get the information you\u0027re looking for.","wrap":true},{"type":"ColumnSet","columns":[{"type":"Column","width":"auto","items":[{"type":"Image","size":"small","url":"https://static.powerapps.com/resource/ppcr/releases/v1.0.1814/1.0.1814.4754/databricks/icon.png","altText":"Azure Databricks"}]},{"type":"Column","width":"stretch","items":[{"type":"TextBlock","size":"medium","weight":"bolder","text":"Azure Databricks","wrap":true}]}]},{"type":"TextBlock","text":"This connection can:","wrap":true},{"type":"TextBlock","id":"truncatedText0","text":"\r- Azure Databricks Genie","wrap":true},{"type":"ColumnSet","columns":[{"type":"Column","width":"auto","items":[{"type":"Image","size":"small","url":"data:image/svg\u002Bxml;base64,PHN2ZyB3aWR0aD0iMTAiIGhlaWdodD0iMTAiIHZpZXdCb3g9IjAgMCAxMCAxMCIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj48cGF0aCBkPSJNNC41IDUuNUM0LjUgNS4yMjM4NiA0LjcyMzg2IDUgNSA1QzUuMjc2MTQgNSA1LjUgNS4yMjM4NiA1LjUg

In [0]:
%skip
# ── RESET: wipe checkpoints + truncate target tables ──────────────────
# Run this before re-ingesting when the CSV parser settings have changed.
#   - Deleting the checkpoint forces Auto Loader to reprocess ALL source files.
#   - Truncating the table removes previously misread rows so they are not
#     duplicated when the corrected ingestion appends new data.
# ⚠️  This is destructive and cannot be undone without re-running ingestion.

for entity_name in entities:
    checkpoint_path = f"{checkpoint_volume_path}/{entity_name}"
    target_table    = f"{target_catalog}.{target_schema}.{entity_name}"

    # 1 — Delete the Auto Loader checkpoint directory
    try:
        dbutils.fs.rm(checkpoint_path, recurse=True)
        print(f"[✓] [{entity_name}] Checkpoint deleted : {checkpoint_path}")
    except Exception:
        print(f"[-] [{entity_name}] No checkpoint found (first run?) — skipping")

    # 2 — Truncate the target Delta table (keeps schema, removes all rows)
    if spark.catalog.tableExists(target_table):
        spark.sql(f"TRUNCATE TABLE {target_table}")
        print(f"[✓] [{entity_name}] Target table truncated : {target_table}")
    else:
        print(f"[-] [{entity_name}] Table does not exist yet — nothing to truncate")

print("\nReset complete — run the Auto Loader cell below to re-ingest with corrected CSV parsing.")

In [0]:
# Auto Loader reads only new CSV partition files on each run (exactly-once via checkpoint).
# trigger(availableNow=True) behaves like a batch job: processes all pending files then stops.
# The checkpoint replaces the old manual log table.
from pyspark.sql import functions as F

for entity_name, entity_schema in entity_schemas.items():
    source_path      = f"{cdm_volume_path}/{entity_name}"
    checkpoint_path  = f"{checkpoint_volume_path}/{entity_name}"
    target_table     = f"{target_catalog}.{target_schema}.{entity_name}"

    print(f"[{entity_name}] source : {source_path}")
    print(f"[{entity_name}] target : {target_table}")
    print(f"[{entity_name}] checkpoint: {checkpoint_path}")

    (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
            .option("header", "false")          # CDM partition CSVs have no column headers
            .option("pathGlobFilter", "*.csv")              # skip manifest/schema JSON files
            .option("multiLine", "true")          # content column contains multi-line JSON
            .option("quote", '"')               # Dataverse CSV: outer quote char is "
            .option("escape", '"')              # Dataverse CSV: inner quotes doubled ("") not backslash-escaped
            .schema(entity_schema)              # schema from model.json — no inference overhead
            .load(source_path)
        .withColumn("_ingested_at",  F.current_timestamp())
        .withColumn("_source_file",  F.col("_metadata.file_path"))   # lineage: which partition CSV
        .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)         # batch-style: finish all pending files then stop
            .toTable(target_table)              # creates UC Delta table on first run
    ).awaitTermination()

    count = spark.table(target_table).count()
    print(f"[{entity_name}] Done. Total rows in target: {count:,}\n")

[conversationtranscript] source : /Volumes/bronze/default/dataverse/conversationtranscript
[conversationtranscript] target : silver.dataverse.conversationtranscript
[conversationtranscript] checkpoint: /Volumes/bronze/default/dataverse/_checkpoints/conversationtranscript
[conversationtranscript] Done. Total rows in target: 524

[systemuser] source : /Volumes/bronze/default/dataverse/systemuser
[systemuser] target : silver.dataverse.systemuser
[systemuser] checkpoint: /Volumes/bronze/default/dataverse/_checkpoints/systemuser
[systemuser] Done. Total rows in target: 318



### Step 5 — Verify Ingested Tables
Iterates over the `entities` list and displays the first 10 rows of each target Delta table (`{target_catalog}.{target_schema}.{entity_name}`). Run this after ingestion to spot-check column alignment, data types, and that the `_ingested_at` / `_source_file` audit columns are populated before pointing downstream consumers at the table.

In [0]:
for entity_name in entities:
    target_table = f"{target_catalog}.{target_schema}.{entity_name}"
    print(f"── {target_table} ──")
    display(spark.table(target_table).orderBy("_ingested_at", ascending=False).limit(10))

── silver.dataverse.conversationtranscript ──


Id SinkCreatedOn SinkModifiedOn statecode statuscode bot_conversationtranscriptid bot_conversationtranscriptid_entitytype createdby createdby_entitytype createdonbehalfby createdonbehalfby_entitytype modifiedby modifiedby_entitytype modifiedonbehalfby modifiedonbehalfby_entitytype owningbusinessunit owningbusinessunit_entitytype owningteam owningteam_entitytype owninguser owninguser_entitytype ownerid ownerid_entitytype bot_conversationtranscriptidname content conversationstarttime conversationtranscriptid createdbyname createdbyyominame createdon createdonbehalfbyname createdonbehalfbyyominame importsequencenumber metadata modifiedbyname modifiedbyyominame modifiedon modifiedonbehalfbyname modifiedonbehalfbyyominame name overriddencreatedon owneridname owneridtype owneridyominame owningbusinessunitname schematype schemaversion timezoneruleversionnumber utcconversiontimezonecode versionnumber _ingested_at _source_file ad8cbd43-bec2-4fce-b685-8cf524d74b4f null null 0 1 e0330f86-c5dd-f011-8544-7c1e525c0f8f bot c2a984f1-5b7e-f011-b4cc-000d3a18ef6f systemuser null null c2a984f1-5b7e-f011-b4cc-000d3a18ef6f systemuser null null ed81698f-467e-f011-b4cc-000d3a18ef6f businessunit null null c2a984f1-5b7e-f011-b4cc-000d3a18ef6f systemuser c2a984f1-5b7e-f011-b4cc-000d3a18ef6f systemuser F1 Fabric Agent {"activities":[{"valueType":"ConversationInfo","type":"trace","timestamp":1782923702,"timestampMs":1782923702634,"from":{"id":"","role":0},"value":{"lastSessionOutcome":"Resolved","lastSessionOutcomeReason":"Resolved","isDesignMode":false,"locale":"en-us"}},{"id":"669c155e-821e-4533-b169-b0767b238ef2","type":"event","timestamp":1782923702,"timestampMs":1782923702634,"from":{"id":"d77e50ca-e5f8-b42f-7d80-881be0485da1","aadObjectId":"e9c36c90-108f-4285-aa64-42d5f1b80a8e","role":1},"name":"pvaSetContext","channelId":"m365copilot","attachments":[]},{"id":"0c38629c-1a08-40fe-9de9-f5add15cd817","type":"message","timestamp":1782923705,"timestampMs":1782923705078,"from":{"id":"d77e50ca-e5f8-b42f-7d80-881be0485da1","aadObjectId":"e9c36c90-108f-4285-aa64-42d5f1b80a8e","role":1},"channelId":"m365copilot","text":"Which drivers have the most wins in the last 10 years?","attachments":[]},{"id":"aa11ebaa-c164-41a3-8bcd-3ac203522239","type":"event","timestamp":1782923708,"timestampMs":1782923708231,"from":{"id":"5d705e9a-4ac1-6249-10d2-931cc6887dbd","role":0},"name":"DynamicServerError","channelId":"m365copilot","attachments":[],"replyToId":"0c38629c-1a08-40fe-9de9-f5add15cd817","value":{"reasonCode":"RequestFailure","errorMessage":"Connector request failed","HttpStatusCode":"notFound","errorResponse":{"errorCode":"CapacityNotActive","isRetriable":false,"message":"Internal error CapacityNotActive.Capacity 6BE8FD38-9FBF-47EB-8A78-E0B3B6569288 is not active","requestId":"24bd112b-66c0-4dfd-8b05-e657cab44506"},"mcpMethod":"initialize","connectorId":"/providers/Microsoft.PowerApps/apis/shared_fabricdataagent","operationId":"InvokeMCP","dialogSchemaName":"auto_agent_svVNg.action.f1_agent"}},{"valueType":"DynamicPlanReceived","id":"1182fcd8-46bf-4382-ae74-ace53fc3d112","type":"event","timestamp":1782923709,"timestampMs":1782923709449,"from":{"id":"5d705e9a-4ac1-6249-10d2-931cc6887dbd","role":0},"name":"DynamicPlanReceived","channelId":"m365copilot","attachments":[],"replyToId":"0c38629c-1a08-40fe-9de9-f5add15cd817","value":{"steps":["P:UniversalSearchTool"],"isFinalPlan":false,"planIdentifier":"e192201d-8c08-426c-9fc0-ed203f24635e"}},{"id":"423d8937-e95b-41f5-a22d-d5dc5844659b","type":"event","timestamp":1782923709,"timestampMs":1782923709449,"from":{"id":"5d705e9a-4ac1-6249-10d2-931cc6887dbd","role":0},"name":"DynamicPlanReceivedDebug","channelId":"m365copilot","attachments":[],"replyToId":"0c38629c-1a08-40fe-9de9-f5add15cd817","value":{"summary":"","ask":"Which drivers have the most wins in the last 10 years?","planIdentifier":"e192201d-8c08-426c-9fc0-ed203f24635e","isFinalPlan":false}},{"valueType":"DynamicPlanStepTriggered","id":"a8a596ee-1975-4c0a-a852-dab9

── silver.dataverse.systemuser ──


Id,SinkCreatedOn,SinkModifiedOn,accessmode,address1_addresstypecode,address1_shippingmethodcode,address2_addresstypecode,address2_shippingmethodcode,azurestate,caltype,deletedstate,emailrouteraccessapproval,incomingemaildeliverymethod,invitestatuscode,outgoingemaildeliverymethod,preferredaddresscode,preferredemailcode,preferredphonecode,systemmanagedusertype,defaultfilterspopulated,displayinserviceviews,isactivedirectoryuser,isallowedbyipfirewall,isdisabled,isemailaddressapprovedbyo365admin,isintegrationuser,islicensed,issyncwithdirectory,setupuser,businessunitid,businessunitid_entitytype,calendarid,calendarid_entitytype,createdby,createdby_entitytype,createdonbehalfby,createdonbehalfby_entitytype,defaultmailbox,defaultmailbox_entitytype,mobileofflineprofileid,mobileofflineprofileid_entitytype,modifiedby,modifiedby_entitytype,modifiedonbehalfby,modifiedonbehalfby_entitytype,parentsystemuserid,parentsystemuserid_entitytype,positionid,positionid_entitytype,queueid,queueid_entitytype,territoryid,territoryid_entitytype,transactioncurrencyid,transactioncurrencyid_entitytype,activedirectoryguid,address1_addressid,address1_city,address1_composite,address1_country,address1_county,address1_fax,address1_latitude,address1_line1,address1_line2,address1_line3,address1_longitude,address1_name,address1_postalcode,address1_postofficebox,address1_stateorprovince,address1_telephone1,address1_telephone2,address1_telephone3,address1_upszone,address1_utcoffset,address2_addressid,address2_city,address2_composite,address2_country,address2_county,address2_fax,address2_latitude,address2_line1,address2_line2,address2_line3,address2_longitude,address2_name,address2_postalcode,address2_postofficebox,address2_stateorprovince,address2_telephone1,address2_telephone2,address2_telephone3,address2_upszone,address2_utcoffset,applicationid,applicationiduri,azureactivedirectoryobjectid,azuredeletedon,businessunitidname,createdbyname,createdbyyominame,createdon,createdonbehalfbyname,createdonbehalfbyyominame,defaultmailboxname,defaultodbfoldername,disabledreason,domainname,employeeid,entityimage,entityimage_timestamp,entityimage_url,entityimageid,exchangerate,firstname,fullname,governmentid,homephone,identityid,importsequencenumber,internalemailaddress,jobtitle,lastname,latestupdatetime,middlename,mobilealertemail,mobileofflineprofileidname,mobilephone,modifiedbyname,modifiedbyyominame,modifiedon,modifiedonbehalfbyname,modifiedonbehalfbyyominame,nickname,organizationid,organizationidname,overriddencreatedon,parentsystemuseridname,parentsystemuseridyominame,passporthi,passportlo,personalemailaddress,photourl,positionidname,processid,queueidname,salutation,sharepointemailaddress,skills,stageid,systemuserid,territoryidname,timezoneruleversionnumber,title,transactioncurrencyidname,traversedpath,userlicensetype,userpuid,utcconversiontimezonecode,versionnumber,windowsliveid,yammeremailaddress,yammeruserid,yomifirstname,yomifullname,yomilastname,yomimiddlename,IsDelete,_ingested_at,_source_file
c54005c7-dd73-f111-ab0f-7c1e525b7a20,null,null,0,1,1,1,1,0,7,0,2,2,0,2,1,1,1,0,false,false,null,false,false,false,false,true,true,false,ed81698f-467e-f011-b4cc-000d3a18ef6f,businessunit,c0d1de83-d91f-415c-910c-edcadd5bb01e,calendar,null,null,null,null,c94005c7-dd73-f111-ab0f-7c1e525b7a20,mailbox,null,null,null,null,null,null,null,null,null,null,cc4005c7-dd73-f111-ab0f-7c1e525b7a20,queue,null,null,null,null,null,68f91807-0f30-4cbc-bbb6-4099f4ed7b6b,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,36032cd5-a114-4f78-8c5b-42287a018edc,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,a9024606-0173-4d32-b3df-4698e3373a20,null,contosojbendermngenvdefault,null,null,2026-06-29T17:13:00.000Z,null,null,James Xu,CRM,null,xujames@MngEnvMCAP738982.onmicrosoft.com,null,null,null,null,null,null,James,James Xu,null,null,156,null,xujames@MngEnvMCAP738982.onmicrosoft.com,null,Xu,null,null,null,null